In [88]:
import pandas as pd

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
print(df.head())

   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies        Contract Pape

In [89]:
print(df.shape)

(7043, 21)


In [90]:
print(df.columns)

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='str')


In [91]:
print(df.describe())

       SeniorCitizen       tenure  MonthlyCharges
count    7043.000000  7043.000000     7043.000000
mean        0.162147    32.371149       64.761692
std         0.368612    24.559481       30.090047
min         0.000000     0.000000       18.250000
25%         0.000000     9.000000       35.500000
50%         0.000000    29.000000       70.350000
75%         0.000000    55.000000       89.850000
max         1.000000    72.000000      118.750000


In [92]:
print(df.isnull().sum())

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


In [93]:
print(df.duplicated())

0       False
1       False
2       False
3       False
4       False
        ...  
7038    False
7039    False
7040    False
7041    False
7042    False
Length: 7043, dtype: bool


In [94]:
# 1. Make a copy of the dataframe
clean_df = df.copy()

# 2. Define the binary columns
features = [
    "Partner", "Dependents", "PhoneService", 
    "OnlineSecurity", "OnlineBackup", "DeviceProtection", 
    "TechSupport", "StreamingTV", "StreamingMovies", "PaperlessBilling"
]

# 3. Standardise strings to lowercase across the selected columns
clean_df[features] = clean_df[features].apply(lambda x: x.str.lower())

# 4. Map "yes"/"no" directly to numerical 1/0
# This handles entries like "no internet service" by leaving them untouched 
# until you decide how to encode them.
clean_df[features] = clean_df[features].replace({"yes": 1, "no": 0})


In [95]:
print(clean_df.head(2))

   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0       1          0       1            0   
1  5575-GNVDE    Male              0       0          0      34            1   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL              0  ...                0   
1                No             DSL              1  ...                1   

  TechSupport StreamingTV StreamingMovies        Contract PaperlessBilling  \
0           0           0               0  Month-to-month                1   
1           0           0               0        One year                0   

      PaymentMethod MonthlyCharges  TotalCharges Churn  
0  Electronic check          29.85         29.85    No  
1      Mailed check          56.95        1889.5    No  

[2 rows x 21 columns]


In [96]:
print(clean_df['Contract'].unique())

<ArrowStringArray>
['Month-to-month', 'One year', 'Two year']
Length: 3, dtype: str


In [97]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import pandas as pd

# ==========================================
# 1. ORDINAL ENCODING (For Rank-Based Columns)
# ==========================================
contract_encoder = OrdinalEncoder(categories=[['Month-to-month', 'One year', 'Two year']])
clean_df['Contract_encoder'] = contract_encoder.fit_transform(clean_df[['Contract']])

# ==========================================
# 2. ONE-HOT ENCODING (For Unordered Categories)
# ==========================================
oh_encoder = OneHotEncoder(sparse_output=False)

# --- FIX: Encoding gender (Correcting the variable name typo) ---
# Note: Double brackets [['gender']] ensure it passes as a 2D array
gender_encoded = oh_encoder.fit_transform(clean_df[['gender']])
encoded_cols_gender = oh_encoder.get_feature_names_out(['gender'])
gender_df = pd.DataFrame(gender_encoded, columns=encoded_cols_gender, index=clean_df.index)

# --- Encoding InternetService ---
internet_encoded = oh_encoder.fit_transform(clean_df[['InternetService']])
encoded_cols_internet = oh_encoder.get_feature_names_out(['InternetService'])
internet_df = pd.DataFrame(internet_encoded, columns=encoded_cols_internet, index=clean_df.index)

# --- Encoding MultipleLines ---
lines_encoded = oh_encoder.fit_transform(clean_df[['MultipleLines']])
encoded_cols_lines = oh_encoder.get_feature_names_out(['MultipleLines'])
lines_df = pd.DataFrame(lines_encoded, columns=encoded_cols_lines, index=clean_df.index)

# --- Encoding PaymentMethod ---
payment_encoded = oh_encoder.fit_transform(clean_df[['PaymentMethod']])
encoded_cols_payment = oh_encoder.get_feature_names_out(['PaymentMethod'])
payment_df = pd.DataFrame(payment_encoded, columns=encoded_cols_payment, index=clean_df.index)

# ==========================================
# 3. MERGING EVERYTHING BACK TOGETHER
# ==========================================
# Save into a fresh df_encoded variable to prevent running into DuplicateError loops
df_encoded = pd.concat([clean_df, gender_df, internet_df, lines_df, payment_df], axis=1)

# ==========================================
# 4. DATA CLEANUP (Noise & String Reduction)
# ==========================================
# Step A: Drop the raw columns we manually encoded
encoded_to_drop = ['gender', 'Contract', 'InternetService', 'MultipleLines', 'PaymentMethod']

# Step B: Drop remaining text columns that haven't been processed yet
# (This stops the "could not convert string to float" errors completely)
remaining_text_cols = ['Partner', 'Dependents', 'PhoneService', 'OnlineSecurity', 
                       'OnlineBackup', 'DeviceProtection', 'TechSupport', 
                       'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'TotalCharges']

df_encoded = df_encoded.drop(columns=encoded_to_drop + remaining_text_cols)

In [98]:
print(df_encoded.head())

   customerID  SeniorCitizen  tenure  MonthlyCharges Churn  Contract_encoder  \
0  7590-VHVEG              0       1           29.85    No               0.0   
1  5575-GNVDE              0      34           56.95    No               1.0   
2  3668-QPYBK              0       2           53.85   Yes               0.0   
3  7795-CFOCW              0      45           42.30    No               1.0   
4  9237-HQITU              0       2           70.70   Yes               0.0   

   gender_Female  gender_Male  InternetService_DSL  \
0            1.0          0.0                  1.0   
1            0.0          1.0                  1.0   
2            0.0          1.0                  1.0   
3            0.0          1.0                  1.0   
4            1.0          0.0                  0.0   

   InternetService_Fiber optic  InternetService_No  MultipleLines_No  \
0                          0.0                 0.0               0.0   
1                          0.0                 0.0

In [102]:
# ==========================================
# 5. SEPARATING FEATURES AND TARGET (FIXED)
# ==========================================
# Force the Churn target column to lowercase, map to 1/0, and explicitly cast as integer
df_encoded['Churn'] = df_encoded['Churn'].astype(str).str.lower().replace({"yes": 1, "no": 0})
df_encoded['Churn'] = df_encoded['Churn'].astype(int)

# Extract your clean numeric target array
y = df_encoded['Churn']

# Isolate features by dropping the target and unique customer identifier
X = df_encoded.drop(columns=['Churn', 'customerID'])

# ==========================================
# 6. TRAIN-TEST SPLIT
# ==========================================
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

# ==========================================
# 7. TRAINING THE MODEL
# ==========================================
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=10000, random_state=0)
clf.fit(X_train, y_train)

print("Model successfully trained! Your target y is now a certified binary integer.")


Model successfully trained! Your target y is now a certified binary integer.
